In [1]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links

In [8]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'expertise / curriculum',
   'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter/X profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook page',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [10]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 8 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'blog page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'professional network',
   'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook page',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [11]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [12]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 15 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
tencent/Hy3
Updated
4 days ago
•
5.57k
•
617
empero-ai/Qwythos-9B-Claude-Mythos-5-1M-GGUF
Updated
11 days ago
•
1.88M
•
1.94k
zai-org/GLM-5.2
Updated
8 days ago
•
362k
•
3.73k
InternScience/Agents-A1
Updated
1 day ago
•
23.1k
•
437
baidu/Unlimited-OCR
Updated
7 days ago
•
1.25M
•
1.9k

In [13]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [14]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [15]:
get_brochure_user_prompt("Hugging Face", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 9 relevant links


'\nYou are looking at a company called: Hugging Face\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\ntencent/Hy3\nUpdated\n4 days ago\n•\n5.57k\n•\n617\nempero-ai/Qwythos-9B-Claude-Mythos-5-1M-GGUF\nUpdated\n11 days 

In [16]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [17]:
create_brochure("Hugging Face", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 8 relevant links


# Hugging Face Brochure

---

## About Hugging Face

Hugging Face is the AI community building the future. It serves as a vibrant collaboration platform where machine learning practitioners come together to create, discover, and share models, datasets, and applications. The platform hosts over 2 million models and more than 500,000 datasets, making it the home of machine learning innovation and open collaboration.

---

## What We Offer

- **Models:** Browse and leverage a vast repository of pretrained models spanning NLP, computer vision, audio, and beyond.
- **Datasets:** Access hundreds of thousands of datasets curated and shared by the global AI community.
- **Spaces:** Deploy and explore AI-powered applications with ease on shared, publicly accessible environments.
- **Buckets:** Manage and store machine learning data and models securely and efficiently.
- **Community:** Engage with a passionate AI community via forums, Discord, blogs, and collaborative projects.
- **Enterprise Solutions:** Tailored support, inference endpoints, and scalable infrastructure for business needs.
- **HuggingChat:** An AI-powered chat solution integrating state-of-the-art conversational capabilities.
- **Learning Resources:** Comprehensive documentation, tutorials, and papers to foster continuous learning.

---

## Our Culture

Hugging Face thrives on openness, collaboration, and community-driven innovation. The company embraces transparency, inclusivity, and ethical AI development, fostering an environment where contributors from around the world can join to push the boundaries of what AI can accomplish. The culture encourages creativity, knowledge-sharing, and rapid experimentation all while supporting responsible AI practices.

---

## Customers & Community

Our platform is trusted by AI researchers, developers, startups, and large enterprises worldwide. With an emphasis on open source and community engagement, Hugging Face empowers users across industries to accelerate their machine learning workflows — from academic research labs to Fortune 500 companies. The diverse user base collaborates on cutting-edge AI across natural language processing, computer vision, speech, reinforcement learning, and more.

---

## Careers at Hugging Face

Join a global team of passionate AI experts, engineers, and community builders committed to creating the best AI tools and community experiences. Hugging Face offers exciting career opportunities that range from software engineering and research to community management and enterprise solutions. If you want to help shape the future of AI in an open, inclusive, and dynamic environment, explore current openings on their careers page.

---

## Why Choose Hugging Face?

- **Unmatched AI Resources:** Access one of the largest and most active repositories of machine learning models and datasets.
- **Collaborative Platform:** Easily share your projects, contribute to open source, and discover other creators’ work.
- **Open Source & Ethical AI:** Strong commitment to transparency, reproducibility, and responsible AI development.
- **Enterprise Ready:** Scalable tools and professional support for production-grade AI deployments.
- **Vibrant Community:** Connect and collaborate with tens of thousands of AI practitioners around the globe.

---

## Contact & Explore

- Visit: [huggingface.co](https://huggingface.co)
- Join the community on Discord and forums
- Explore models, datasets, and Spaces
- Discover careers and contribute to the future of AI

---

Hugging Face – Empowering the AI community to build the future, together.

In [21]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [22]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links


# Hugging Face Brochure

---

## About Hugging Face

Hugging Face is the vibrant AI community and collaboration platform shaping the future of machine learning. Renowned as the home of machine learning, Hugging Face enables practitioners, researchers, and enterprises worldwide to create, discover, and collaborate on models, datasets, and AI applications. With over 2 million hosted models and 500,000 datasets, Hugging Face stands as the leading open-source stack and community hub for advancing AI technology.

---

## What We Offer

- **Models**: Explore and contribute to a diverse and ever-growing collection of more than 2 million pre-trained machine learning models for various AI tasks.
- **Datasets**: Access, share, and collaborate on over half a million datasets spanning different domains to fuel AI research and applications.
- **Spaces**: Develop and deploy AI applications seamlessly with community-powered Spaces, enabling interactive demos and real-time AI-powered apps.
- **Buckets**: Enterprise-grade storage solutions to securely host datasets and models.
- **Enterprise Solutions**: Tailored support including Hugging Face PRO, inference endpoints, and storage buckets for businesses needing scalable AI infrastructure.
- **Community & Learning**: Engage via forums, Discord, blogs, daily research papers, and Github repositories to stay at the cutting edge of AI technology.

---

## Hugging Face Community & Culture

At its core, Hugging Face fosters an open, inclusive, and collaborative culture dedicated to democratizing AI. The company advocates for open source and transparency, supporting a global community of researchers, developers, and enthusiasts who contribute daily to accelerate AI advancements together.

- Emphasis on open collaboration and knowledge sharing.
- Supportive environment nurturing creativity and innovation.
- Active community events and resources including forums, chat, and learning materials.
- Commitment to diversity and making AI accessible to all.

---

## Customers & Use Cases

Hugging Face serves a diverse spectrum of users ranging from independent AI researchers and hobbyists to large enterprises and AI-driven startups. Its platform supports applications such as:

- Natural language processing (chatbots, translation, summarization)
- Computer vision (image recognition, editing, OCR)
- Speech and audio processing (voice chat, voice-to-voice applications)
- Video generation and more

Companies leverage Hugging Face’s models and infrastructure to accelerate AI application development, productionize AI with scalable inference endpoints, and harness community expertise to solve complex problems effectively.

---

## Careers at Hugging Face

Joining Hugging Face means contributing to an open AI ecosystem that shapes the future of machine learning. The company looks for passionate engineers, researchers, product managers, and community builders who:

- Are excited about open source and collaborative innovation
- Want to work on impactful AI projects used by millions worldwide
- Thrive in a fast-moving, inclusive, and creative environment

Roles are available in engineering, research, community engagement, product development, and enterprise solutions. Hugging Face fosters growth, offers meaningful challenges, and values employee autonomy and creativity.

---

## Connect with Hugging Face

- Website: [huggingface.co](https://huggingface.co)
- Community: Join on Discord, Forum, GitHub
- Blog & Research: Stay updated with posts and daily papers
- Explore AI Apps, Models & Datasets directly on the platform

---

## Brand Identity

Hugging Face is represented by its distinctive yellow (#FFD21E) and orange (#FF9D00) colors, symbolizing innovation and energy in AI. The approachable and collaborative brand reflects the company's mission to make AI universally accessible and easy to use.

---

## Summary

Hugging Face is the premier collaborative hub for machine learning — where community and technology meet to build the future of AI. Whether you are a researcher looking to share models, a developer deploying AI applications, or an enterprise scaling machine learning operations, Hugging Face offers the tools, community, and expertise to help you succeed.

Join the future of AI with Hugging Face today!

**Trying Gradio for Brochure generator using the relevant links from the website**

In [33]:
import gradio as gr

In [34]:
def stream_gpt(prompt):
    messages = [
        {"role": "system", "content": brochure_system_prompt},
        {"role": "user", "content": prompt}
      ]
    stream = openai.chat.completions.create(
        model='gpt-4.1-mini',
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [35]:
import requests
requests.get("http://localhost:11434/").content

b'Ollama is running'

In [36]:
ollama_url = "http://localhost:11434/v1"
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

In [37]:
def stream_llama(prompt):
    messages = [
        {"role": "system", "content": brochure_system_prompt},
        {"role": "user", "content": prompt}
      ]
    stream = ollama.chat.completions.create(
        model='llama3.2',
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [38]:
def stream_brochure_new(company_name, url, model):
    yield ""
    #prompt = f"Please generate a company brochure for {company_name}. Here is their landing page:\n"
    #prompt += fetch_website_contents(url)
    
    prompt = get_brochure_user_prompt(company_name, url)
    if model=="GPT":
        result = stream_gpt(prompt)
    elif model=="Llama":
        result = stream_llama(prompt)
    else:
        raise ValueError("Unknown model")
    yield from result

In [ ]:
name_input = gr.Textbox(label="Company name:")
url_input = gr.Textbox(label="Landing page URL including http:// or https://")
model_selector = gr.Dropdown(["GPT", "Llama"], label="Select model", value="GPT")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_brochure_new,
    title="Brochure Generator", 
    inputs=[name_input, url_input, model_selector], 
    outputs=[message_output], 
    examples=[
            ["Hugging Face", "https://huggingface.co", "GPT"],
            ["Edward Donner", "https://edwarddonner.com", "Llama"]
        ], 
    flagging_mode="never"
    )
view.launch(inbrowser = True)

/Users/hrishikeshharishkumar/llm_engineering_course/llm_engineering/.venv/lib/python3.12/site-packages/gradio/components/dropdown.py:230: UserWarning: The value passed into gr.Dropdown() is not in the list of choices. Please update the list of choices to include: Claude or set allow_custom_value=True.
  warnings.warn(


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
